# Pandas 01 — Loading and first inspection

Every section starts with a tiny table you can check by eye, then shows the same
thing on the real file.

**What's in here**
- `read_csv` and what it guesses
- `shape`, `dtypes`, `info()`, `describe()`
- missing values, duplicates, cardinality
- is the timestamp really a timestamp? is the interval constant?
- `read_csv` options worth memorising
- memory, `head` / `tail` / `sample`
- an `inspect(df)` checklist assembled from the pieces above
- writing data out

Data: `../data/hourly_power_raw.csv` (messy, as received) and `../data/hourly_power_clean.csv` (tidy).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. Reading a CSV — what pandas guesses

Start with a 4-line CSV written to a temporary file so every column is visible.
One value in `price` is the word `missing`, and `time` is text.

In [2]:
import tempfile, os

csv_text = """time,load,price
2023-01-01 00:00,10,50.0
2023-01-01 01:00,12,missing
2023-01-01 02:00,11,48.5
2023-01-01 03:00,13,52.0
"""
tmp = os.path.join(tempfile.gettempdir(), "tiny.csv")
with open(tmp, "w") as f:
    f.write(csv_text)

toy = pd.read_csv(tmp)
toy

,time,load,price
0,2023-01-01 00:00,10,50.0
1,2023-01-01 01:00,12,missing
2,2023-01-01 02:00,11,48.5
3,2023-01-01 03:00,13,52.0


It looks fine. Now check the dtypes: `time` is `object` (text) and `price` is `object`
too, because of that one `missing`. Only `load` became a number.

In [3]:
toy.dtypes

time     object
load      int64
price    object
dtype: object

The real file, same call.

In [4]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw.head()

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
0,2023-03-30 23:00:00,28555.6,4.97,8.75,0.0,70.16,GB
1,2023-07-16 15:00:00,27602.0,21.96,6.52,523.2,23.19,GB
2,2022-12-25 16:00:00,33773.8,4.15,9.57,18.5,115.79,GB
3,2023-03-10 01:00:00,26151.0,2.06,5.83,0.0,80.50,GB
4,2023-11-09 05:00:00,24584.6,6.58,4.76,0.0,69.49,GB


In [5]:
raw.dtypes

time                object
consumption_mwh    float64
temp_c             float64
wind_ms            float64
solar_wm2          float64
price_eur_mwh       object
region              object
dtype: object

**Interview check:** *"Why is `price_eur_mwh` an object column?"* — at least one value
could not be parsed as a number. Find it before converting (section 6 of the next notebook).

## 2. `shape`, `dtypes`, `info()`

`shape` is (rows, columns). `info()` shows dtype, non-null count and memory per column.

In [6]:
toy.shape

(4, 3)

In [7]:
toy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   time    4 non-null      object
 1   load    4 non-null      int64 
 2   price   4 non-null      object
dtypes: int64(1), object(2)
memory usage: 224.0+ bytes


In [8]:
print(raw.shape)
raw.info()

(17457, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17457 entries, 0 to 17456
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             17457 non-null  object 
 1   consumption_mwh  17457 non-null  float64
 2   temp_c           17308 non-null  float64
 3   wind_ms          17457 non-null  float64
 4   solar_wm2        17457 non-null  float64
 5   price_eur_mwh    17457 non-null  object 
 6   region           17457 non-null  object 
dtypes: float64(4), object(3)
memory usage: 954.8+ KB


## 3. `describe()`

`describe()` summarises numeric columns. Always read `min` and `max`: a sentinel
like `-999` shows up there immediately.

In [9]:
t = pd.DataFrame({"temp": [5.0, 6.0, -999.0, 7.0]})
t.describe()

,temp
count,4.000000
mean,-245.250000
std,502.500663
min,-999.000000
25%,-246.000000
50%,5.500000
75%,6.250000
max,7.000000


`min = -999` is not a temperature. `include="all"` adds object columns
(count, unique, top, freq).

In [10]:
toy.describe(include="all")

,time,load,price
count,4,4.000000,4
unique,4,NaN,4
top,2023-01-01 00:00,NaN,50.0
freq,1,NaN,1
mean,NaN,11.500000,NaN
std,NaN,1.290994,NaN
min,NaN,10.000000,NaN
25%,NaN,10.750000,NaN
50%,NaN,11.500000,NaN
75%,NaN,12.250000,NaN


In [11]:
raw.describe().round(1)

,consumption_mwh,temp_c,wind_ms,solar_wm2
count,17457.0,17308.0,17457.0,17457.0
mean,29315.5,6.3,7.3,97.7
std,4208.6,61.1,2.5,155.4
min,18092.9,-999.0,0.0,0.0
25%,26446.5,4.4,5.6,0.0
50%,29671.3,9.9,7.2,0.0
75%,32374.4,15.3,9.0,143.1
max,40824.9,27.7,16.0,794.6


`temp_c` has `min = -999`: a sentinel to deal with later.

## 4. Missing values — count and share

`isna()` gives True where a value is missing. Summing Trues counts them; the mean is the share.

In [12]:
m = pd.DataFrame({"a": [1.0, np.nan, 3.0, np.nan], "b": [1, 2, 3, 4]})
m

,a,b
0,1.0,1
1,NaN,2
2,3.0,3
3,NaN,4


In [13]:
m.isna()

,a,b
0,False,False
1,True,False
2,False,False
3,True,False


In [14]:
m.isna().sum()

a    2
b    0
dtype: int64

In [15]:
m.isna().mean()

a    0.5
b    0.0
dtype: float64

Two of four in `a` are missing → 0.5. On the real file:

In [16]:
raw.isna().sum()

time                 0
consumption_mwh      0
temp_c             149
wind_ms              0
solar_wm2            0
price_eur_mwh        0
region               0
dtype: int64

## 5. Duplicates

`duplicated()` marks a row as True when an identical row appeared earlier.

In [17]:
d = pd.DataFrame({"k": ["a", "b", "a", "c"], "v": [1, 2, 1, 3]})
d

,k,v
0,a,1
1,b,2
2,a,1
3,c,3


In [18]:
d.duplicated()

0    False
1    False
2     True
3    False
dtype: bool

Row 2 is `a, 1` again, so it is marked. `.sum()` counts them.
`duplicated(subset=["k"])` looks at the key only.

In [19]:
print("exact duplicates :", d.duplicated().sum())
print("duplicate keys   :", d.duplicated(subset=["k"]).sum())

exact duplicates : 1
duplicate keys   : 1


In [20]:
print("exact duplicate rows in raw :", raw.duplicated().sum())
print("duplicate timestamps in raw :", raw.duplicated(subset=["time"]).sum())

exact duplicate rows in raw : 15
duplicate timestamps in raw : 15


## 6. Cardinality — `nunique()` and `value_counts()`

`nunique()` = how many distinct values per column. A column with 1 distinct value is
constant (useless); one with as many as rows is an id.

In [21]:
c = pd.DataFrame({"region": ["GB", "GB", "GB"], "tariff": ["A", "B", "A"], "id": [1, 2, 3]})
c

,region,tariff,id
0,GB,A,1
1,GB,B,2
2,GB,A,3


In [22]:
c.nunique()

region    1
tariff    2
id        3
dtype: int64

In [23]:
c["tariff"].value_counts()

tariff
A    2
B    1
Name: count, dtype: int64

In [24]:
raw.nunique()

time               17442
consumption_mwh    16497
temp_c              2844
wind_ms             1366
solar_wm2           4062
price_eur_mwh       9932
region                 1
dtype: int64

`region` has 1 distinct value: constant. `time` has fewer distinct values than rows: duplicates.

In [25]:
raw["region"].value_counts()

region
GB    17457
Name: count, dtype: int64

## 7. Is the timestamp really a timestamp?

Text timestamps have dtype `object`, and `.dt` does not work on them.

In [26]:
toy["time"]

0    2023-01-01 00:00
1    2023-01-01 01:00
2    2023-01-01 02:00
3    2023-01-01 03:00
Name: time, dtype: object

In [27]:
try:
    toy["time"].dt.hour
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: Can only use .dt accessor with datetimelike values


`pd.to_datetime` converts. Now the dtype is `datetime64` and `.dt.hour` works.

In [28]:
toy["time"] = pd.to_datetime(toy["time"])
toy["time"]

0   2023-01-01 00:00:00
1   2023-01-01 01:00:00
2   2023-01-01 02:00:00
3   2023-01-01 03:00:00
Name: time, dtype: datetime64[ns]

In [29]:
toy["time"].dt.hour

0    0
1    1
2    2
3    3
Name: time, dtype: int32

In [30]:
raw["time"] = pd.to_datetime(raw["time"])
print(raw["time"].dtype)
print(raw["time"].min(), "->", raw["time"].max())

datetime64[ns]
2022-01-01 00:00:00 -> 2023-12-31 23:00:00


## 8. Is the sampling interval constant?

Sort by time, take the difference between consecutive timestamps, count the differences.
With a clean hourly series there is exactly one value: 1 hour.

In [31]:
g = pd.DataFrame({"time": pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00",
                                          "2023-01-01 03:00", "2023-01-01 03:00"])})
g

,time
0,2023-01-01 00:00:00
1,2023-01-01 01:00:00
2,2023-01-01 03:00:00
3,2023-01-01 03:00:00


In [32]:
g["time"].diff()

0               NaT
1   0 days 01:00:00
2   0 days 02:00:00
3   0 days 00:00:00
Name: time, dtype: timedelta64[ns]

Row 2 jumped 2 hours (an hour is missing), row 3 jumped 0 (a duplicate).
`value_counts()` summarises that.

In [33]:
g["time"].diff().value_counts()

time
0 days 01:00:00    1
0 days 02:00:00    1
0 days 00:00:00    1
Name: count, dtype: int64

In [34]:
raw = raw.sort_values("time")
raw["time"].diff().value_counts()

time
0 days 01:00:00    17387
0 days 02:00:00       52
0 days 00:00:00       15
1 days 01:00:00        1
0 days 03:00:00        1
Name: count, dtype: int64

Mostly 1 hour; 52 gaps of 2 hours and one of 3 hours (missing hours), 15 of 0 (duplicates), one of 25 hours (a whole missing day).

## 9. `read_csv` options worth memorising

Each option shown on the tiny file. `parse_dates` converts while reading.

In [35]:
pd.read_csv(tmp, parse_dates=["time"]).dtypes

time     datetime64[ns]
load              int64
price            object
dtype: object

`na_values` tells pandas which strings mean missing, so `price` becomes a float.

In [36]:
pd.read_csv(tmp, na_values=["missing"])

,time,load,price
0,2023-01-01 00:00,10,50.0
1,2023-01-01 01:00,12,NaN
2,2023-01-01 02:00,11,48.5
3,2023-01-01 03:00,13,52.0


In [37]:
pd.read_csv(tmp, na_values=["missing"]).dtypes

time      object
load       int64
price    float64
dtype: object

`usecols` reads only some columns; `nrows` only some rows (quick look at a huge file).

In [38]:
pd.read_csv(tmp, usecols=["time", "load"], nrows=2)

,time,load
0,2023-01-01 00:00,10
1,2023-01-01 01:00,12


`dtype` forces a type; `index_col` uses a column as the index.

In [39]:
pd.read_csv(tmp, dtype={"load": "float64"}, index_col="time")

,load,price
time,,
2023-01-01 00:00,10.0,50.0
2023-01-01 01:00,12.0,missing
2023-01-01 02:00,11.0,48.5
2023-01-01 03:00,13.0,52.0


European files: `sep=";"`, `decimal=","`, `thousands="."`.

In [40]:
eu = os.path.join(tempfile.gettempdir(), "eu.csv")
with open(eu, "w") as f:
    f.write("time;load\n2023-01-01 00:00;1.234,5\n2023-01-01 01:00;1.300,0\n")
pd.read_csv(eu, sep=";", decimal=",", thousands=".")

,time,load
0,2023-01-01 00:00,1234.5
1,2023-01-01 01:00,1300.0


`chunksize` reads a big file piece by piece (each piece is a DataFrame).

In [41]:
for chunk in pd.read_csv(tmp, chunksize=2):
    print(chunk.shape)

(2, 3)
(2, 3)


## 10. Reading the clean file with timestamps that carry an offset

The clean file's `time` looks like `2022-01-01 00:00:00+00:00`. With `parse_dates`
it becomes timezone-aware (`datetime64[ns, UTC]`).

In [42]:
clean = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
clean.head(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71


In [43]:
clean["time"].dtype

datetime64[ns, UTC]

If you forget `parse_dates`, `pd.to_datetime(..., utc=True)` does the same afterwards.
`errors="coerce"` turns unparseable strings into `NaT` instead of raising.

In [44]:
pd.to_datetime(pd.Series(["2023-01-01", "not a date"]), errors="coerce")

0   2023-01-01
1          NaT
dtype: datetime64[ns]

## 11. Memory

`memory_usage(deep=True)` counts the bytes of strings too. Object columns are the expensive ones.

In [45]:
raw.memory_usage(deep=True)

Index               139656
time                139656
consumption_mwh     139656
temp_c              139656
wind_ms             139656
solar_wm2           139656
price_eur_mwh      1090789
region             1029963
dtype: int64

## 12. `head` / `tail` / `sample`

`head` shows the first rows, `tail` the last, `sample` random ones (fix `random_state`
to get the same rows every time). Look at all three: files are often sorted in a way that hides problems.

In [46]:
raw.head(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
4680,2022-01-01 00:00:00,26858.4,0.11,7.00,0.0,81.83,GB
1864,2022-01-01 01:00:00,26177.8,-0.18,6.61,0.0,88.21,GB
3978,2022-01-01 02:00:00,26229.4,-1.11,7.14,0.0,84.71,GB


In [47]:
raw.tail(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
17147,2023-12-31 21:00:00,30263.2,3.87,7.04,0.0,77.68,GB
10662,2023-12-31 22:00:00,28415.8,3.31,8.23,0.0,85.78,GB
1740,2023-12-31 23:00:00,27054.8,2.39,8.12,0.0,56.82,GB


In [48]:
raw.sample(3, random_state=0)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
5419,2023-05-10 12:00:00,29795.9,14.88,5.88,455.5,57.93,GB
7008,2022-08-11 02:00:00,22430.0,18.28,5.27,0.0,110.60,GB
11337,2022-04-20 14:00:00,28380.2,15.11,5.36,244.5,85.81,GB


## 13. A reusable `inspect(df)` checklist

The pieces above, in one function you can paste into any notebook.

In [49]:
def inspect(df):
    print("shape        :", df.shape)
    print("dtypes       :", df.dtypes.value_counts().to_dict())
    print("missing      :", df.isna().sum()[df.isna().sum() > 0].to_dict())
    print("duplicates   :", df.duplicated().sum())
    print("constant cols:", list(df.columns[df.nunique() <= 1]))
    for col in df.columns:
        if str(df[col].dtype).startswith("datetime"):
            print(f"time column {col}: {df[col].min()} -> {df[col].max()}, "
                  f"unique={df[col].is_unique}, sorted={df[col].is_monotonic_increasing}")

inspect(raw)

shape        : (17457, 7)
dtypes       : {dtype('float64'): 4, dtype('O'): 2, dtype('<M8[ns]'): 1}
missing      : {'temp_c': 149}
duplicates   : 15
constant cols: ['region']
time column time: 2022-01-01 00:00:00 -> 2023-12-31 23:00:00, unique=False, sorted=True


In [50]:
inspect(clean)

shape        : (17520, 6)
dtypes       : {dtype('float64'): 5, datetime64[ns, UTC]: 1}
missing      : {}
duplicates   : 0
constant cols: []
time column time: 2022-01-01 00:00:00+00:00 -> 2023-12-31 23:00:00+00:00, unique=True, sorted=True


## 14. Writing data out

`index=False` avoids writing the row numbers as a column.

In [51]:
out = os.path.join(tempfile.gettempdir(), "out.csv")
toy.to_csv(out, index=False)
print(open(out).read())

time,load,price
2023-01-01 00:00:00,10,50.0
2023-01-01 01:00:00,12,missing
2023-01-01 02:00:00,11,48.5
2023-01-01 03:00:00,13,52.0



Parquet keeps dtypes (timestamps stay timestamps) and is much smaller; it needs `pyarrow`.

In [52]:
try:
    import pyarrow  # noqa
    toy.to_parquet(os.path.join(tempfile.gettempdir(), "out.parquet"))
    print("parquet written")
except ImportError:
    print("pyarrow not installed - skip")

pyarrow not installed - skip


## Summary — questions before any modelling

1. What is one row? (one hour? one meter-day?)
2. Are the dtypes right? (timestamps parsed, numbers numeric)
3. How many missing values, and where?
4. Any duplicate rows or duplicate keys?
5. Any constant columns, any id-like columns?
6. Time range, and is the interval constant?
7. `describe()`: do min / max make physical sense?